In [ ]:
import rawpy
import numpy as np

raw = rawpy.imread("rawboat.ARW")

# Bayer brut du capteur (zone visible, sans les marges optiques)
bayer = raw.raw_image_visible.astype(np.float32)   # (H, W) uint16 -> float32

# Metadonnees utiles pour l'ISP
print("shape            :", bayer.shape)
print("min / max        :", bayer.min(), bayer.max())
print("bayer pattern    :", raw.color_desc, raw.raw_pattern.tolist())
print("black level      :", raw.black_level_per_channel)
print("white level      :", raw.white_level)
print("camera WB        :", raw.camera_whitebalance)
print("XYZ->cam matrix  :\n", raw.rgb_xyz_matrix)

In [ ]:
import matplotlib.pyplot as plt

# --- Etape 1 : black level + normalisation dans [0, 1] ---
black = raw.black_level_per_channel[0]     # 800
white = raw.white_level                     # 16383
norm  = np.clip((bayer - black) / (white - black), 0, 1)

plt.figure(figsize=(8, 5))
plt.imshow(norm, cmap="gray")
plt.title("Etape 1 : Bayer brut normalise (1 seule matrice, mosaique RGGB)")
plt.axis("off"); plt.show()

In [ ]:
# --- Etape 2 : balance des blancs ---
# gains mesures par la camera (R, G, B), normalises sur le vert
wb = np.array(raw.camera_whitebalance[:3], np.float32)
wb = wb / wb[1]
print("gains WB (R, G, B) :", wb.round(3))

# on multiplie chaque pixel par le gain correspondant a SA couleur
# raw_colors_visible : 0=R, 1=G, 2=B, 3=G2
lut  = np.array([wb[0], wb[1], wb[2], wb[1]], np.float32)
gain = lut[raw.raw_colors_visible]
wb_bayer = np.clip(norm * gain, 0, 1)

plt.figure(figsize=(8, 5))
plt.imshow(wb_bayer, cmap="gray")
plt.title("Etape 2 : Bayer apres balance des blancs")
plt.axis("off"); plt.show()

In [ ]:
# --- Etape 3 : demosaicing simple (1 bloc 2x2 RGGB -> 1 pixel RGB) ---
# motif RGGB :  R=(0,0)  G1=(0,1)  G2=(1,0)  B=(1,1)
R  = wb_bayer[0::2, 0::2]
G1 = wb_bayer[0::2, 1::2]
G2 = wb_bayer[1::2, 0::2]
B  = wb_bayer[1::2, 1::2]
G  = (G1 + G2) / 2

rgb = np.stack([R, G, B], axis=-1)        # (H/2, W/2, 3)  <- enfin 3 canaux !
print("image RGB :", rgb.shape)

plt.figure(figsize=(8, 5))
plt.imshow(np.clip(rgb, 0, 1))
plt.title("Etape 3 : apres demosaicing (RGB lineaire, couleurs pas encore corrigees)")
plt.axis("off"); plt.show()

In [ ]:
# --- Etape 4 : correction couleur (capteur -> sRGB lineaire) ---
# rgb_xyz_matrix = matrice XYZ -> capteur. On veut l'inverse capteur -> sRGB.
cam_xyz = raw.rgb_xyz_matrix[:3, :3]                    # XYZ -> capteur
srgb2xyz = np.array([[0.4124, 0.3576, 0.1805],
                     [0.2126, 0.7152, 0.0722],
                     [0.0193, 0.1192, 0.9505]], np.float32)  # sRGB lin -> XYZ

cam_srgb = cam_xyz @ srgb2xyz                           # sRGB lin -> capteur
cam_srgb = cam_srgb / cam_srgb.sum(1, keepdims=True)    # normalise (neutre -> neutre)
cam2rgb = np.linalg.inv(cam_srgb)                       # capteur -> sRGB lin

rgb_corr = np.clip(rgb @ cam2rgb.T, 0, 1)

plt.figure(figsize=(8, 5))
plt.imshow(rgb_corr)
plt.title("Etape 4 : apres correction couleur (sRGB lineaire)")
plt.axis("off"); plt.show()

In [ ]:
# --- Etape 5 : gamma sRGB -> image finale affichable ---
def to_srgb(x):
    return np.where(x <= 0.0031308, 12.92 * x, 1.055 * np.power(x, 1 / 2.4) - 0.055)

final = np.clip(to_srgb(rgb_corr), 0, 1)

plt.figure(figsize=(10, 6))
plt.imshow(final)
plt.title("Image finale sRGB")
plt.axis("off"); plt.show()

# sauvegarde optionnelle
import imageio.v3 as iio
iio.imwrite("out_srgb.png", (final * 255).astype(np.uint8))